# Comprehensive Guide to Dialogue Systems
## A Practical Tutorial — From Classical to Modern Approaches

This notebook is a step-by-step tutorial for building a dialogue system from scratch.
Each section progressively adds capabilities, culminating in a fully multilingual,
semantically aware conversational agent.

| Section | Topic |
|---------|-------|
| 1 | Setup: Library installation & Translation Layer |
| 2 | Classical Theory: GUS Architecture & Frames |
| 3 | Implementing `DialogueFrame` |
| 4 | Natural Language Understanding (NLP) |
| 5 | Dialogue Management (`DialogueManager`) |
| 6 | Real Service Handlers + Live Weather API |
| 7 | Semantic Intent Detection |
| 8 | Multilingual Agent |
| 9 | Interactive Chat |


## Section 1: Setup & Installation

### Required Libraries
| Library | Purpose |
|---------|--------|
| `re` | Regular expressions for text pattern matching (Python built-in) |
| `requests` | HTTP calls to the live weather API |
| `deep-translator` | Free translation to/from any language |
| `langdetect` | Automatic language detection from text |

> **Important:** Run the cell below first to install all dependencies.


In [1]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

libs = {'deep-translator': 'deep_translator', 'langdetect': 'langdetect', 'requests': 'requests'}
for pip_name, import_name in libs.items():
    try:
        __import__(import_name)
        print(f'  OK  {pip_name}')
    except ImportError:
        print(f'  Installing {pip_name}...')
        install(pip_name)
        print(f'  Done {pip_name}')

import re, random, requests
from deep_translator import GoogleTranslator
from langdetect import detect as detect_lang
print('\nAll libraries loaded.')


  OK  deep-translator
  OK  langdetect
  OK  requests

All libraries loaded.


### 1.1 The Translation Layer — Foundation of Everything

The most important architectural principle in this system:

```
User Input  (any language)
    ↓  Detect language
    ↓  Translate to English
    ↓  Internal processing (English only)
    ↓  Translate response back to user's language
    ↓  Display
```

With this design, **all** internal keywords, patterns, and logic are written in English only.
Users can write in Persian, Swedish, German, Arabic, Ukrainian, or any other supported language.

**Supported languages:** Persian, English, Swedish, German, French, Arabic, Spanish, Ukrainian


In [2]:
class TranslationLayer:
    """Detects language, translates input to EN, translates output back.
    Uses session-language locking so short words like 'لندن' are not
    misclassified between Persian / Arabic / Urdu.
    """

    SUPPORTED = {
        'fa': 'Persian', 'en': 'English', 'sv': 'Swedish',
        'de': 'German',  'fr': 'French',  'ar': 'Arabic',
        'es': 'Spanish', 'uk': 'Ukrainian'
    }
    # Languages that share the Arabic script and are easily confused
    ARABIC_SCRIPT = {'fa', 'ar', 'ur'}

    def __init__(self):
        self.session_lang = None   # locked after first long message
        self.user_lang    = 'en'

    # Persian-exclusive characters that never appear in pure Arabic text
    PERSIAN_ONLY_CHARS = set('چپگژ')

    def _detect(self, text):
        """Detect language with session-lock and Persian character override."""
        try:
            raw = detect_lang(text)
            # Always map Urdu → Persian (same script, Persian preferred)
            raw = 'fa' if raw == 'ur' else raw
        except Exception:
            raw = self.session_lang or 'en'

        # ── Persian character override ─────────────────────────────────────
        # If text contains Persian-exclusive letters (چ پ گ ژ), it MUST be
        # Persian regardless of what the detector says.
        if any(ch in text for ch in self.PERSIAN_ONLY_CHARS):
            raw = 'fa'

        # ── Short text: trust session_lang ────────────────────────────────
        if len(text.strip()) < 15 and self.session_lang:
            if raw in self.ARABIC_SCRIPT and self.session_lang in self.ARABIC_SCRIPT:
                return self.session_lang

        # ── Long text: update session_lang ───────────────────────────────
        if len(text.strip()) >= 15:
            self.session_lang = raw

        return raw

    def to_english(self, text):
        self.user_lang = self._detect(text)
        if self.user_lang == 'en':
            return text
        try:
            translated = GoogleTranslator(
                source=self.user_lang, target='en').translate(text)
            lang_name = self.SUPPORTED.get(self.user_lang, self.user_lang)
            print(f'  [Translate] {lang_name} → EN: "{translated}"')
            return translated
        except Exception as e:
            print(f'  [Translate Error] GoogleTranslator failed. Check internet/VPN or rate limits: {e}')
            return text

    def from_english(self, text):
        if self.user_lang == 'en':
            return text
        try:
            return GoogleTranslator(
                source='en', target=self.user_lang).translate(text)
        except Exception as e:
            print(f'  [Translate Error] GoogleTranslator failed. Check internet/VPN or rate limits: {e}')
            return text


# Quick test
tl = TranslationLayer()
test_long  = 'فردا می‌خواهم به استکهلم بروم، چه لباسی بپوشم؟'
test_short = 'لندن'  # short word — would normally mis-detect as Arabic
tl.to_english(test_long)   # sets session_lang = 'fa'
en_short = tl.to_english(test_short)
print(f'Short word  : {test_short}  →  {en_short}  (lang used: {tl.user_lang})')
print('Session lock test passed!' if tl.user_lang == 'fa' else 'WARNING: session lock failed')


  [Translate] Persian → EN: "I want to go to Stockholm tomorrow, what should I wear?"
  [Translate] Persian → EN: "London"
Short word  : لندن  →  London  (lang used: fa)
Session lock test passed!


---
## Section 2: Classical Architecture — GUS and the Frame Concept

The **GUS** system (Genial Understander System, Bobrow et al., 1977) was one of the first
task-oriented dialogue systems, developed at Xerox PARC to book airline tickets via natural
language conversation.

### Core Ideas
- Each **task** (e.g., booking a flight) is represented as a **Frame** — a structured record
- Each frame has named **Slots** that must be filled (e.g., `location`, `date`)
- The system asks questions to fill empty slots, one by one (**Slot Filling**)
- Users can provide multiple slot values in one utterance (**Mixed Initiative**)

### Legacy
GUS's frame-based dialogue management is the direct predecessor of modern systems like
Google Dialogflow, Amazon Lex, and RASA — and is what we implement in this notebook.

> **Library required:** Standard Python only — no special imports needed for this section.


---
## Section 3: Implementing the Frame (`DialogueFrame`)

Each service (Weather, Restaurant, Transport) is represented as a `DialogueFrame`.
The frame tracks which information slots have been filled and which are still missing.

> **Library required:** Standard Python OOP — no imports needed.


In [3]:
class DialogueFrame:
    """Represents one service task with named slots to fill."""

    def __init__(self, name, slots):
        self.name = name
        self.slots = {slot: None for slot in slots}

    def is_complete(self):
        return all(v is not None for v in self.slots.values())

    def get_missing_slot(self):
        return next((s for s, v in self.slots.items() if v is None), None)

    def reset(self):
        self.slots = {s: None for s in self.slots}

    def __repr__(self):
        return f'Frame({self.name}, {self.slots})'


def get_initial_frames():
    return {
        'weather':    DialogueFrame('Weather',    ['location', 'date']),
        'restaurant': DialogueFrame('Restaurant', ['cuisine', 'location', 'price_range']),
        'transport':  DialogueFrame('Transport',  ['origin', 'destination', 'time']),
    }

frames = get_initial_frames()
print('Frames created:')
for name, f in frames.items():
    print(f'  {name}: slots = {list(f.slots.keys())}')


Frames created:
  weather: slots = ['location', 'date']
  restaurant: slots = ['cuisine', 'location', 'price_range']
  transport: slots = ['origin', 'destination', 'time']


---
## Section 4: Natural Language Understanding (NLP)

Because the **Translation Layer** converts all input to English first,
every keyword and pattern can be written in **English only** — simpler and more maintainable.

We extract two types of information from each user message:
- **Intent**: Which service does the user want? (weather / restaurant / transport)
- **Entity**: Specific values — which city? which date? what cuisine?

> **Library required: `re`** — regex search with `re.IGNORECASE` and combined patterns via `'|'.join(...)`


In [4]:
# ── English-only keywords (translation layer handles other languages) ──────
KEYWORDS = {
    'intents': {
        'weather':    ['weather', 'forecast', 'temperature', 'climate'],
        'restaurant': ['restaurant', 'food', 'eat', 'hungry', 'dinner', 'lunch', 'meal', 'cafe'],
        'transport':  ['bus', 'train', 'flight', 'ticket', 'travel', 'commute', 'trip'],
    },
    'entities': {
        'location':    ['london', 'tehran', 'paris', 'stockholm', 'berlin', 'amsterdam',
                        'rome', 'madrid', 'dubai', 'tokyo', 'new york', 'sydney',
                        'gothenburg', 'vienna', 'zurich', 'oslo', 'copenhagen'],
        'date':        ['today', 'tomorrow', 'monday', 'tuesday', 'wednesday',
                        'thursday', 'friday', 'saturday', 'sunday', 'weekend', 'next week'],
        'cuisine':     ['italian', 'pizza', 'persian', 'kebab', 'chinese', 'japanese',
                        'indian', 'mexican', 'french', 'sushi', 'burger', 'thai'],
        'price_range': ['cheap', 'budget', 'affordable', 'expensive', 'luxury', 'fine dining'],
        'origin':      ['home', 'airport', 'hotel', 'station', 'office'],
        'destination': ['work', 'office', 'city center', 'downtown', 'airport', 'hotel', 'station'],
        'time':        ['morning', 'afternoon', 'evening', 'night', '8am', '10am', '12pm', '8pm'],
    }
}

def extract_info(text):
    """Extract intent and entities from English text using regex."""
    found_intent = None
    found_entities = {}

    for intent, words in KEYWORDS['intents'].items():
        if re.search('|'.join(re.escape(w) for w in words), text, re.IGNORECASE):
            found_intent = intent
            break

    for etype, examples in KEYWORDS['entities'].items():
        m = re.search('|'.join(re.escape(e) for e in examples), text, re.IGNORECASE)
        if m:
            found_entities[etype] = m.group(0).lower()

    return found_intent, found_entities


### Test: extract_info in action
The cell below tests our NLP function on three example sentences:


In [5]:
# ── Test extract_info ────────────────────────────────────────────────────────
tests = [
    'I want cheap pizza in stockholm tomorrow',
    'What is the weather forecast for berlin this weekend?',
    'Book a flight from airport to city center in the morning',
]
print('NLP Test Results:')
print('-' * 55)
for t in tests:
    intent, entities = extract_info(t)
    print(f'  Input:    {t}')
    print(f'  Intent:   {intent}')
    print(f'  Entities: {entities}')
    print()


NLP Test Results:
-------------------------------------------------------
  Input:    I want cheap pizza in stockholm tomorrow
  Intent:   None
  Entities: {'location': 'stockholm', 'date': 'tomorrow', 'cuisine': 'pizza', 'price_range': 'cheap'}

  Input:    What is the weather forecast for berlin this weekend?
  Intent:   weather
  Entities: {'location': 'berlin', 'date': 'weekend'}

  Input:    Book a flight from airport to city center in the morning
  Intent:   transport
  Entities: {'origin': 'airport', 'destination': 'airport', 'time': 'morning'}



---
## Section 5: Distributional Semantics — The Term-Document Matrix

### Theory (Chapter 24, Jurafsky & Martin)

The key idea of **distributional semantics** is:

> *"You shall know a word by the company it keeps."* — J.R. Firth (1957)

Instead of hand-crafted rules, we represent **words** and **intents** as vectors in a shared
geometric space. The angle (cosine similarity) between two vectors tells us how closely related they are.

#### Step 1: Term-Document Matrix

We build a matrix where:
- **Rows** = vocabulary words  
- **Columns** = intent classes (weather / restaurant / transport)  
- **Cells** = how often that word appears in training sentences for that intent

```
              | weather | restaurant | transport
 -------------|---------|------------|----------
 wear         |   5     |     0      |    0
 coat         |   4     |     0      |    0
 cold         |   3     |     0      |    1
 eat          |   0     |     5      |    0
 food         |   0     |     4      |    0
 train        |   0     |     0      |    4
```

#### Step 2: TF-IDF Weighting

Raw counts give common words (like "the") too much weight.
**TF-IDF** (Term Frequency × Inverse Document Frequency) downweights common words and
highlights discriminative ones.

#### Step 3: LSA — Latent Semantic Analysis

We apply **SVD** (Singular Value Decomposition) to reduce the matrix to *k* dimensions,
capturing **latent (hidden) semantic relationships** between words and intents.

A query *"what should I bring to Tehran?"* might not contain "weather", but after
SVD it lands close to the **weather** column — because "bring" and "pack" co-occur with
weather-related training sentences.

In [6]:
import numpy as np
from collections import defaultdict
import re as _re

# ─── Training corpus (intent → list of example sentences) ───────────────────
INTENT_CORPUS = {
    "weather": [
        "what should I wear tomorrow",
        "will it be cold in berlin",
        "what clothes should I pack for stockholm",
        "is it going to rain in paris",
        "what is the weather like in london",
        "do I need a coat for amsterdam",
        "what jacket should I bring to oslo",
        "is it warm in tokyo",
        "what temperature in new york tomorrow",
        "umbrella needed in rome",
        "forecast for tehran",
        "what to wear in vienna next week",
        "do I need sunscreen in cairo",
        "will it snow in helsinki",
        "what clothes to bring for trip to madrid",
        "dress code for cold winter in moscow",
        "what weather expected in dubai",
        "should I pack warm clothes for budapest",
        "any rain expected in prague",
        "how hot will it be in istanbul",
    ],
    "restaurant": [
        "I am hungry find me a restaurant",
        "where can I eat pizza in london",
        "best sushi place nearby",
        "cheap food in berlin",
        "good Chinese restaurant in paris",
        "I need lunch recommendations",
        "where to dine in amsterdam",
        "best place to eat in rome",
        "find me a kebab shop",
        "vegetarian food near me",
        "I am starving what to eat",
        "recommend a nice dinner place",
        "italian restaurant in vienna",
        "fast food near the hotel",
        "halal restaurant in london",
    ],
    "transport": [
        "how do I get from airport to city",
        "bus from the station to center",
        "train ticket to downtown",
        "need a ride to the hotel",
        "subway map for berlin",
        "how to get to the conference",
        "taxi from airport",
        "public transport in stockholm",
        "metro line to the museum",
        "ferry to the island",
        "bus schedule morning",
        "ride share from airport",
    ],
}

def _tokenize(text):
    """Lowercase, remove punctuation, split into tokens."""
    return _re.sub(r"[^a-z ]", " ", text.lower()).split()

def build_term_document_matrix(corpus):
    """
    Build a term × intent count matrix from the training corpus.
    Returns: (vocab list, intent list, raw count matrix ndarray)
    """
    intents = list(corpus.keys())
    # Count word occurrences per intent
    counts = defaultdict(lambda: defaultdict(int))
    for intent, sentences in corpus.items():
        for sent in sentences:
            for tok in _tokenize(sent):
                counts[tok][intent] += 1

    vocab = sorted(counts.keys())
    matrix = np.array([[counts[w][intent] for intent in intents]
                        for w in vocab], dtype=float)

    print(f"Term-Document Matrix: {matrix.shape[0]} words × {matrix.shape[1]} intents")
    print(f"Intents: {intents}")
    print(f"Sample (top-10 discriminative rows):")

    # Show top-10 rows by variance (most discriminative)
    variances = matrix.var(axis=1)
    top_idx   = variances.argsort()[-10:][::-1]
    col_w     = max(len(i) for i in intents) + 2
    header    = f"{'word':15s}" + "".join(f"{i:>{col_w}}" for i in intents)
    print(header)
    print("-" * len(header))
    for idx in top_idx:
        row_str = f"{vocab[idx]:15s}" + "".join(f"{int(matrix[idx,j]):>{col_w}}" for j in range(len(intents)))
        print(row_str)

    return vocab, intents, matrix

VOCAB, INTENTS, RAW_MATRIX = build_term_document_matrix(INTENT_CORPUS)
print("\nTerm-Document Matrix built successfully.")

Term-Document Matrix: 120 words × 3 intents
Intents: ['weather', 'restaurant', 'transport']
Sample (top-10 discriminative rows):
word                weather  restaurant   transport
---------------------------------------------------
in                       13           7           1
what                      8           1           0
for                       6           0           1
it                        5           0           0
i                         6           4           1
to                        5           3           8
the                       1           1           5
should                    4           0           0
restaurant                0           4           0
from                      0           0           4

Term-Document Matrix built successfully.


### Applying LSA — Singular Value Decomposition (SVD)

LSA compresses the term-document matrix by factorizing it:

```
M ≈ U · Σ · Vᵀ
```

- **U** = word vectors in latent space (shape: words × k)  
- **Σ** = singular values (importance of each dimension)  
- **Vᵀ** = intent vectors in latent space (shape: k × intents)  

We keep only the top **k** dimensions (we use k=5 here since we have only 3 intents —
for a real system k=100–300 is common as shown in Chapter 24).

A **query vector** is computed the same way as a document vector: sum of its word vectors.  
Then we compute **cosine similarity** to each intent centroid to find the closest intent.

```
            query
              │
     project into LSA space
              │
    ┌─────────┼─────────┐
    │         │         │
weather  restaurant  transport
  0.87      0.03       0.10   ← cosine similarity scores
    │
 → Intent: weather  ✓
```

In [7]:
def _tfidf_transform(matrix):
    import warnings
    """Apply log-frequency and IDF normalization (Chapter 6 style)."""
    # Log frequency: w = 1 + log(tf)  if tf > 0  else 0
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        log_matrix = np.where(matrix > 0, 1 + np.log(matrix), 0)
    # IDF: log(N/df) where N=num intents, df=num intents where word appears
    N  = matrix.shape[1]
    df = (matrix > 0).sum(axis=1, keepdims=True).astype(float)
    df = np.where(df == 0, 1, df)         # avoid div-by-zero
    idf = np.log(N / df)
    return log_matrix * idf

class LSAIntentClassifier:
    """
    Latent Semantic Analysis classifier for dialogue intent detection.

    Based on: Foltz, Kintsch & Landauer (1998) — Chapter 24 reference.
    Method:
      1. Build term-document matrix from training corpus
      2. Apply TF-IDF weighting
      3. Apply SVD → keep top-k singular vectors (LSA space)
      4. Project new query into LSA space
      5. Cosine similarity to each intent column → argmax = predicted intent
    """

    def __init__(self, corpus, k=5):
        self.intents = list(corpus.keys())
        self.vocab, _, raw = build_term_document_matrix(corpus)
        self.word2idx = {w: i for i, w in enumerate(self.vocab)}

        tfidf_mat  = _tfidf_transform(raw)           # (words × intents)

        # Thin SVD: M ≈ U Σ Vᵀ
        # k_eff ≤ k because we can only have as many singular values as min(rows, cols)
        self.k_eff = min(k, min(tfidf_mat.shape))
        U, s, Vt   = np.linalg.svd(tfidf_mat, full_matrices=False)
        self.Uk    = U[:, :self.k_eff]               # (words   × k_eff)
        self.Sk    = s[:self.k_eff]                   # (k_eff,)
        self.Vkt   = Vt[:self.k_eff, :]              # (k_eff   × intents)

        # Each intent is already a column in the LSA space
        self.intent_vecs = (np.diag(self.Sk) @ self.Vkt).T  # (intents × k_eff)
        print(f"LSA model built: {self.k_eff} latent dimensions, {len(self.vocab)} vocab words")

    def _query_vector(self, tokens):
        """Sum of LSA word vectors for tokens present in vocabulary."""
        vec = np.zeros(self.k_eff)
        for tok in tokens:
            if tok in self.word2idx:
                row_idx = self.word2idx[tok]
                # Project word vector into LSA space: Uk[row] * Sk
                vec += self.Uk[row_idx, :] * self.Sk
        return vec

    def _cosine(self, a, b):
        denom = (np.linalg.norm(a) * np.linalg.norm(b))
        return float(np.dot(a, b) / denom) if denom > 1e-9 else 0.0

    def predict(self, text, verbose=True):
        """
        Predict intent for a query string.
        Returns (intent_name, {intent: score}) or (None, {}) if all scores < threshold.
        """
        tokens = _tokenize(text)
        q_vec  = self._query_vector(tokens)

        if np.linalg.norm(q_vec) < 1e-9:
            # No known words at all — cannot classify
            return None, {}

        scores = {intent: self._cosine(q_vec, self.intent_vecs[i])
                  for i, intent in enumerate(self.intents)}

        best_intent = max(scores, key=scores.get)
        best_score  = scores[best_intent]

        if verbose:
            score_str = "  ".join(f"{k}:{v:.2f}" for k, v in scores.items())
            print(f"  [LSA] query={repr(text)}")
            print(f"        scores: {score_str}")
            print(f"        → intent: {best_intent} (cos={best_score:.2f})")

        # Confidence threshold: if best score < 0.15 → unknown
        if best_score < 0.15:
            return None, scores
        return best_intent, scores


# Build the global LSA classifier (used by all future managers)
LSA_CLASSIFIER = LSAIntentClassifier(INTENT_CORPUS, k=5)
print("\nLSA Intent Classifier ready.")

Term-Document Matrix: 120 words × 3 intents
Intents: ['weather', 'restaurant', 'transport']
Sample (top-10 discriminative rows):
word                weather  restaurant   transport
---------------------------------------------------
in                       13           7           1
what                      8           1           0
for                       6           0           1
it                        5           0           0
i                         6           4           1
to                        5           3           8
the                       1           1           5
should                    4           0           0
restaurant                0           4           0
from                      0           0           4
LSA model built: 3 latent dimensions, 120 vocab words

LSA Intent Classifier ready.


### Demo: LSA in Action

Watch how the LSA classifier handles **implicit** queries — sentences that never
contain "weather", "restaurant", or "transport" directly.

In [8]:
print("=" * 65)
print("Demo: LSA Intent Classification — Implicit Queries")
print("=" * 65)

test_queries = [
    # Weather (implicit)
    "I want to go to Tehran in two days, what clothes should I bring?",
    "heading to Berlin this weekend — hot or cold?",
    "do I need an umbrella in Oslo next friday?",
    # Restaurant (implicit)
    "I am starving, any good sushi place in London?",
    "where should we dine tonight in paris?",
    # Transport (implicit)
    "how do I reach the city center from the airport?",
    "need a ride to the hotel in the morning",
]

for q in test_queries:
    intent, scores = LSA_CLASSIFIER.predict(q, verbose=True)
    print()

Demo: LSA Intent Classification — Implicit Queries
  [LSA] query='I want to go to Tehran in two days, what clothes should I bring?'
        scores: weather:1.00  restaurant:0.05  transport:0.01
        → intent: weather (cos=1.00)

  [LSA] query='heading to Berlin this weekend — hot or cold?'
        scores: weather:1.00  restaurant:0.01  transport:0.01
        → intent: weather (cos=1.00)

  [LSA] query='do I need an umbrella in Oslo next friday?'
        scores: weather:1.00  restaurant:0.01  transport:0.11
        → intent: weather (cos=1.00)

  [LSA] query='I am starving, any good sushi place in London?'
        scores: weather:0.19  restaurant:0.98  transport:0.00
        → intent: restaurant (cos=0.98)

  [LSA] query='where should we dine tonight in paris?'
        scores: weather:0.67  restaurant:0.75  transport:0.01
        → intent: restaurant (cos=0.75)

  [LSA] query='how do I reach the city center from the airport?'
        scores: weather:0.14  restaurant:0.00  transport:0

---
## Section 6: Entity Grid — Tracking Discourse Entities Across Turns

### Theory (Barzilay & Lapata, 2005 — cited in Chapter 24)

An **Entity Grid** is a 2-D matrix that captures **which entities appear in which
dialogue turns** and what grammatical role they play:

```
Turn │ location │  date   │ topic
─────┼──────────┼─────────┼──────────
  1  │  S (Tehran) │ O (two days) │  –
  2  │     –    │    –    │ S (clothes)
  3  │     O    │    –    │  –
```

- **S** = Subject  
- **O** = Object / complement  
- **X** = Other mention  
- **–** = Not mentioned  

**Why does this help in dialogue?**

If the user already mentioned `location=Tehran` in turn 1, the system should **not ask**
for the location again in turn 2. The Entity Grid makes this *carryover* explicit.

### Entity Types We Track

| Entity type | Examples |
|-------------|---------|
| `location`  | city names, countries |
| `date`      | tomorrow, next friday, in two days |
| `cuisine`   | pizza, sushi, kebab |
| `time`      | morning, 8am, evening |
| `clothing`  | coat, umbrella, jacket, wear |
| `topic`     | weather, food, transport |

In [9]:
class EntityGrid:
    """
    Tracks entity mentions across dialogue turns.

    Each cell contains the grammatical role of the entity in that turn:
       "S" = Subject, "O" = Object/complement, "–" = absent
    
    Based on: Barzilay & Lapata (2005) — Chapter 24, Section 24.2
    """

    ROLE_SUBJECT = "S"
    ROLE_OBJECT  = "O"
    ROLE_ABSENT  = "–"

    ENTITY_PATTERNS = {
        # Locations: common city names + generic
        "location": _re.compile(
            r"\b(tehran|stockholm|berlin|london|paris|amsterdam|rome|tokyo|"
            r"new\s+york|oslo|vienna|cairo|budapest|prague|istanbul|dubai|"
            r"moscow|madrid|helsinki|copenhagen|zurich|bangkok|singapore)\b", _re.I),
        # Dates
        "date": _re.compile(
            r"\b(today|tomorrow|yesterday|monday|tuesday|wednesday|thursday|"
            r"friday|saturday|sunday|next\s+week|this\s+weekend|"
            r"in\s+(?:one|two|three|four|five|six|seven|\d+)\s+days?|"
            r"\d{1,2}/\d{1,2}(?:/\d{2,4})?)\b", _re.I),
        # Cuisine / food
        "cuisine": _re.compile(
            r"\b(pizza|sushi|kebab|chinese|italian|indian|french|thai|"
            r"japanese|mexican|vegetarian|halal|burger|pasta|seafood)\b", _re.I),
        # Time of day
        "time": _re.compile(
            r"\b(morning|afternoon|evening|night|midnight|noon|"
            r"\d{1,2}\s*(?:am|pm)|\d{1,2}:\d{2})\b", _re.I),
        # Clothing concepts → implies weather intent
        "clothing": _re.compile(
            r"\b(wear|wearing|coat|jacket|umbrella|clothes|clothing|dress|"
            r"outfit|raincoat|boots|sweater|pack|bring|take|luggage)\b", _re.I),
    }

    def __init__(self):
        self.turns   = []          # list of {entity_type: role}
        self.known   = {}          # entity_type → most recent value

    def add_turn(self, text):
        """Process a user turn and update the entity grid."""
        turn_row = {etype: self.ROLE_ABSENT for etype in self.ENTITY_PATTERNS}

        for etype, pattern in self.ENTITY_PATTERNS.items():
            m = pattern.search(text)
            if m:
                val   = m.group(0).lower().strip()
                # Heuristic: subject if near start of sentence, else object
                role  = self.ROLE_SUBJECT if m.start() < len(text) // 2 else self.ROLE_OBJECT
                turn_row[etype] = role
                self.known[etype] = val          # remember value for slot carryover

        self.turns.append(turn_row)
        return turn_row

    def get_carried_slots(self):
        """
        Return slots that have already been mentioned (S or O) in
        previous turns → these do NOT need to be asked again.
        """
        return dict(self.known)   # copy of all seen entity values

    def print_grid(self):
        """Pretty-print the entity grid (for tutorial illustration)."""
        etypes = list(self.ENTITY_PATTERNS.keys())
        col_w  = 12
        header = f"{'Turn':>5} │" + "".join(f"{e:>{col_w}}" for e in etypes)
        print(header)
        print("─" * len(header))
        for t, row in enumerate(self.turns):
            cells = "".join(
                f"{(row[e] + ' (' + self.turns[t].get(e,'') + ')') if row[e] != '–' else '–':>{col_w}}"
                for e in etypes)
            # simpler version:
            cells = "".join(f"{row[e]:>{col_w}}" for e in etypes)
            print(f"{t+1:>5} │{cells}")


print("EntityGrid class defined.")

# ──────────────────────────────────────────────────────────────────────────────
# Demo: Entity Grid for a multi-turn conversation
# ──────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Demo: Entity Grid for Multi-Turn Dialogue")
print("=" * 60)

grid = EntityGrid()

sample_turns = [
    "In two days I want to travel to Tehran, what clothes should I bring?",
    "I am thinking of a warm coat and maybe an umbrella",
    "Could you also find a good restaurant near the hotel?",
]

for i, turn in enumerate(sample_turns):
    print(f"\nTurn {i+1}: {repr(turn)}")
    row = grid.add_turn(turn)
    filled = {k: v for k, v in row.items() if v != "–"}
    print(f"  Entities detected: {filled}")

print("\nFull Entity Grid:")
grid.print_grid()
print("\nCarried slots (available for slot-filling without asking again):")
print(grid.get_carried_slots())

EntityGrid class defined.

Demo: Entity Grid for Multi-Turn Dialogue

Turn 1: 'In two days I want to travel to Tehran, what clothes should I bring?'
  Entities detected: {'location': 'S', 'date': 'S', 'clothing': 'O'}

Turn 2: 'I am thinking of a warm coat and maybe an umbrella'
  Entities detected: {'clothing': 'S'}

Turn 3: 'Could you also find a good restaurant near the hotel?'
  Entities detected: {}

Full Entity Grid:
 Turn │    location        date     cuisine        time    clothing
───────────────────────────────────────────────────────────────────
    1 │           S           S           –           –           O
    2 │           –           –           –           –           S
    3 │           –           –           –           –           –

Carried slots (available for slot-filling without asking again):
{'location': 'tehran', 'date': 'in two days', 'clothing': 'coat'}


---
## Section 7: Combined Intent Detection — LSA + Entity Grid + Rules

### Architecture

We now **combine all three methods** into a unified intent detection pipeline:

```
User Input (English)
       │
       ▼
 ┌─────────────────────────────────────────────────┐
 │  Step 1: Clothing Override (safety rule)        │
 │  If "wear / coat / umbrella / pack" → weather   │
 └─────────────┬───────────────────────────────────┘
               │ (if no override)
               ▼
 ┌─────────────────────────────────────────────────┐
 │  Step 2: LSA Cosine Similarity                  │
 │  Project query → LSA space → argmax intent      │
 └─────────────┬───────────────────────────────────┘
               │ (if LSA score < 0.15 = uncertain)
               ▼
 ┌─────────────────────────────────────────────────┐
 │  Step 3: Keyword Fallback                       │
 │  Original concept-map matching                  │
 └─────────────┬───────────────────────────────────┘
               │
               ▼
 ┌─────────────────────────────────────────────────┐
 │  Step 4: Entity Grid — Slot Carryover           │
 │  Extract entities using pattern matching        │
 │  Slots seen in previous turns → auto-filled     │
 └─────────────────────────────────────────────────┘
               │
               ▼
      Slot Filling → Service Call → Response
```

### Translation Layer (preserved)
The **TranslationLayer** remains the outermost wrapper:
- Input → `to_english()` before any processing  
- Output → `from_english()` before showing to user

### Relative Date Parser

Before the dialogue manager can call the weather service,
it must convert relative date phrases like *"in two days"*
or *"next friday"* into actual calendar dates.

This function is used inside `LSADialogueManager` when the user
provides a date answer to the slot-filling question.

In [10]:
from datetime import datetime, timedelta

# ── Relative date parser ─────────────────────────────────────────────────────
_NUM_WORDS = {
    'one':1,'two':2,'three':3,'four':4,'five':5,'six':6,'seven':7,
    'eight':8,'nine':9,'ten':10,'a':1,'an':1,'couple':2,'few':3,
}
_DAYS = {'monday':0,'tuesday':1,'wednesday':2,'thursday':3,
         'friday':4,'saturday':5,'sunday':6}

def _to_int(word):
    try: return int(word)
    except (ValueError, TypeError): return _NUM_WORDS.get(str(word).lower())

def parse_relative_date(text):
    """
    Convert relative date phrase (English) to a resolved date string.
    Examples:
      "tomorrow"      → "2026-03-08"
      "in two days"   → "2026-03-09"
      "next friday"   → "2026-03-13"
      "this weekend"  → "2026-03-07"
    Returns None if no relative date pattern is found.
    """
    import re
    text = text.lower().strip()
    today = datetime.now().date()

    if 'today' in text:
        return str(today)
    if 'tomorrow' in text:
        return str(today + timedelta(days=1))
    if 'yesterday' in text:
        return str(today - timedelta(days=1))
    if 'weekend' in text:
        # next Saturday
        days_ahead = (5 - today.weekday()) % 7
        return str(today + timedelta(days=days_ahead or 7))

    # "in N days" / "N days from now"
    m = re.search(r'in\s+(\w+)\s+days?', text)
    if m:
        n = _to_int(m.group(1))
        if n:
            return str(today + timedelta(days=n))

    # "next <weekday>"
    m = re.search(r'next\s+(monday|tuesday|wednesday|thursday|friday|saturday|sunday)', text)
    if m:
        target = _DAYS[m.group(1)]
        days_ahead = (target - today.weekday() + 7) % 7 or 7
        return str(today + timedelta(days=days_ahead))

    # bare weekday name
    for day_name, day_num in _DAYS.items():
        if day_name in text:
            days_ahead = (day_num - today.weekday() + 7) % 7 or 7
            return str(today + timedelta(days=days_ahead))

    return None

print("parse_relative_date() ready.")
print(f"  'tomorrow'    → {parse_relative_date('tomorrow')}")
print(f"  'in two days' → {parse_relative_date('in two days')}")
print(f"  'next friday' → {parse_relative_date('next friday')}")
print(f"  'this weekend'→ {parse_relative_date('this weekend')}")

parse_relative_date() ready.
  'tomorrow'    → 2026-03-09
  'in two days' → 2026-03-10
  'next friday' → 2026-03-13
  'this weekend'→ 2026-03-14


In [11]:
# ─── Natural-language slot questions (unchanged from before) ─────────────────
SLOT_QUESTIONS = {
    'location':    'Which city are you asking about?',
    'date':        'What date? (e.g. today, tomorrow, friday)',
    'cuisine':     'What type of food? (e.g. pizza, sushi, kebab)',
    'price_range': 'What is your budget? (cheap / moderate / luxury)',
    'origin':      'Where are you starting from? (e.g. home, airport)',
    'destination': 'Where do you want to go? (e.g. city center, office)',
    'time':        'What time? (e.g. morning, 8am, evening)',
}

# ─── Updated extract_info using LSA + Entity Grid ────────────────────────────
def lsa_extract_info(text, entity_grid=None):
    """
    Unified intent + entity extraction using:
      1. Clothing safety override
      2. LSA cosine similarity
      3. Keyword fallback
      4. Entity Grid pattern matching
    Returns: (intent_str_or_None, entities_dict)
    """
    lower = text.lower()
    tokens = _tokenize(text)

    # ── Step 1: Clothing override ─────────────────────────────────────────
    clothing_words = {'wear', 'wearing', 'coat', 'jacket', 'umbrella',
                      'clothes', 'clothing', 'outfit', 'dress', 'pack', 'bring'}
    if clothing_words & set(tokens):
        intent = 'weather'
        print(f'  [Override] Clothing word detected → intent=weather')
    else:
        # ── Step 2: LSA classification ────────────────────────────────────
        intent, scores = LSA_CLASSIFIER.predict(text, verbose=True)

    # ── Step 3: Keyword fallback if LSA uncertain ─────────────────────────
    if intent is None:
        KEYWORD_MAP = {
            'weather':    {'weather','forecast','temperature','rain','snow','hot','cold','warm','wear','clothes'},
            'restaurant': {'restaurant','eat','food','hungry','starving','dine','lunch','dinner'},
            'transport':  {'bus','train','taxi','metro','subway','flight','travel','ride','transport','trip','go'},
        }
        for intent_name, kws in KEYWORD_MAP.items():
            if kws & set(tokens):
                intent = intent_name
                print(f'  [Keyword fallback] matched {kws & set(tokens)} → intent={intent}')
                break

    # ── Step 4: Entity extraction via EntityGrid patterns ─────────────────
    entities = {}
    if entity_grid is not None:
        turn_row = entity_grid.add_turn(text)
        # Start from carryover (previously seen slots)
        entities = dict(entity_grid.get_carried_slots())
        # Map entity types → slot names
        ENTITY_TO_SLOT = {'location': 'location', 'date': 'date',
                          'cuisine': 'cuisine', 'time': 'time'}
        for etype, slot in ENTITY_TO_SLOT.items():
            if etype in entities:
                print(f'  [EntityGrid] slot {slot} = "{entities[etype]}" (from grid)')
    else:
        # Standalone entity extraction (no grid context)
        tmp_grid = EntityGrid()
        tmp_grid.add_turn(text)
        entities = tmp_grid.get_carried_slots()
        ENTITY_TO_SLOT = {'location': 'location', 'date': 'date',
                          'cuisine': 'cuisine', 'time': 'time'}
        entities = {ENTITY_TO_SLOT[k]: v for k, v in entities.items()
                    if k in ENTITY_TO_SLOT}

    # Map price keywords
    price_map = {'cheap': 'cheap', 'budget': 'cheap', 'expensive': 'luxury',
                 'luxury': 'luxury', 'moderate': 'moderate', 'affordable': 'cheap'}
    for kw, val in price_map.items():
        if kw in lower:
            entities['price_range'] = val

    return intent, entities


class LSADialogueManager:
    """
    Frame-based Dialogue Manager upgraded with:
    - LSA intent classification (distributional semantics, Ch. 24)
    - Entity Grid for cross-turn slot carryover (Ch. 24)
    - Relative date parsing
    - Keyword fallback safety net
    """

    def __init__(self, frames, handlers):
        self.frames       = frames
        self.handlers     = handlers
        self.active_frame = None
        self.entity_grid  = EntityGrid()          # ← tracks all entities across turns

    def process_input(self, english_text):
        intent, entities = lsa_extract_info(english_text, self.entity_grid)

        # Activate frame on new intent
        if intent and intent in self.frames:
            if not self.active_frame or intent != self.active_frame.name.lower():
                self.active_frame = self.frames[intent]
                self.active_frame.reset()
                print(f'  [System] Service activated: {self.active_frame.name}')

        if not self.active_frame:
            return 'Hello! I can help with weather, restaurants, or transport.'

        # ── Direct Slot Assignment (for short follow-up replies) ─────────────
        missing   = self.active_frame.get_missing_slot()
        words     = english_text.strip().split()
        no_intent = intent is None
        if missing and missing not in entities and no_intent and 1 <= len(words) <= 4:
            if missing == 'date':
                resolved = parse_relative_date(english_text)
                entities['date'] = resolved if resolved else english_text.strip().lower()
                print(f'  [Direct] date = "{entities["date"]}')
            else:
                entities[missing] = english_text.strip().lower()
                print(f'  [Direct] {missing} = "{entities[missing]}"')

        # Fill slots from extracted entities
        for etype, val in entities.items():
            if etype in self.active_frame.slots:
                self.active_frame.slots[etype] = val

        if self.active_frame.is_complete():
            fname     = self.active_frame.name.lower()
            slots_now = dict(self.active_frame.slots)
            self.active_frame.reset()
            self.active_frame = None
            return self.handlers.get(fname, lambda s: str(s))(slots_now)

        missing = self.active_frame.get_missing_slot()
        return SLOT_QUESTIONS.get(missing, f"Please specify '{missing}'.")


# For backward compatibility: alias
SemanticDialogueManager = LSADialogueManager
print('LSADialogueManager (LSA + EntityGrid) ready.')

LSADialogueManager (LSA + EntityGrid) ready.


---
## Section 8: Real Service Handlers

### 8.1 Live Weather API (OpenWeatherMap)
To receive real weather data, get a free API key from [openweathermap.org](https://openweathermap.org):
1. Create a free account
2. Copy your API key from the **API Keys** dashboard
3. Paste it into the `OWM_API_KEY` variable below

> If no key is provided, the system automatically falls back to **simulated (mock) data**.

### 8.2 Restaurant & Transport
In a real project these would call the **Google Places API** and a **GTFS/transit API**.
For this tutorial they return randomly generated plausible data.

In [12]:
# ── OpenWeatherMap API key ───────────────────────────────────────────────────
# اگر کلید ندارید، خالی بگذارید — سیستم از Mock استفاده می‌کند
OWM_API_KEY = ''  # ← کلید خود را اینجا بگذارید

def service_weather(slots):
    location = slots.get('location', 'london')
    date     = slots.get('date', 'today')

    # ── Real API call ────────────────────────────────────────────────────────
    if OWM_API_KEY:
        try:
            url = (f'https://api.openweathermap.org/data/2.5/weather'
                   f'?q={location}&appid={OWM_API_KEY}&units=metric')
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                d    = r.json()
                temp = d['main']['temp']
                desc = d['weather'][0]['description']
                feel = d['main']['feels_like']
                if temp < 5:
                    advice = 'Wear a heavy coat and gloves.'
                elif temp < 12:
                    advice = 'Wear a warm jacket.'
                elif 'rain' in desc:
                    advice = 'Take an umbrella.'
                else:
                    advice = 'Dress comfortably.'
                return (f'[Live] Weather in {location.title()} ({date}): '
                        f'{temp:.1f}°C (feels like {feel:.1f}°C) | '
                        f'{desc.capitalize()} | {advice}')
            else:
                print(f'  [OWM] API error {r.status_code}, using mock data.')
        except Exception as e:
            print(f'  [OWM] Connection error: {e}, using mock data.')

    # ── Mock fallback ────────────────────────────────────────────────────────
    conditions = [
        ('sunny',  22, 'Dress comfortably.'),
        ('cloudy', 14, 'Wear an extra layer.'),
        ('rainy',  10, 'Take an umbrella.'),
        ('snowy',  -2, 'Wear a heavy coat and gloves.'),
    ]
    cond, temp, advice = random.choice(conditions)
    return (f'[Mock] Weather in {location.title()} ({date}): '
            f'{temp}°C | {cond.capitalize()} | {advice}')


def service_restaurant(slots):
    cuisine  = slots.get('cuisine', 'local')
    location = slots.get('location', 'city center')
    price    = slots.get('price_range', 'moderate')
    options  = {
        'pizza':    ['Pizza Palace', 'Napoli Star', 'Roma Express'],
        'kebab':    ['Kebab King', 'Persian Grill', 'Shiraz Garden'],
        'italian':  ['La Bella Italia', 'Venezia Ristorante', 'Trattoria Roma'],
        'sushi':    ['Tokyo Garden', 'Sakura', 'Zen Kitchen'],
        'burger':   ['Burger Lab', 'The Grill House', 'Stack & Smash'],
    }
    chosen = random.choice(options.get(cuisine, ['The Local Bistro', 'City Eats']))
    stars  = random.randint(4, 5)
    return (f'[Mock] Best {cuisine} restaurant in {location.title()} ({price}): '
            f'{chosen} | Rating: {stars}/5 ⭐')


def service_transport(slots):
    origin   = slots.get('origin', 'your location')
    dest     = slots.get('destination', 'destination')
    time_val = slots.get('time', 'any time')
    price    = random.randint(2, 20)
    duration = random.randint(10, 60)
    return (f'[Mock] Route: {origin} → {dest} | '
            f'Departure: {time_val} | Duration: {duration} min | '
            f'Price: €{price}')


SERVICE_HANDLERS = {
    'weather':    service_weather,
    'restaurant': service_restaurant,
    'transport':  service_transport,
}

# Quick weather test
dm_test = LSADialogueManager(get_initial_frames(), SERVICE_HANDLERS)
print('Service handlers ready.')
if OWM_API_KEY:
    print('  OWM API key found — will use live weather data.')
else:
    print('  No OWM API key — will use mock weather data.')


Service handlers ready.
  No OWM API key — will use mock weather data.


### Demo: LSA + Entity Grid in Action

The system now correctly handles the Tehran/clothing query that triggered this upgrade.

In [13]:
print("=" * 65)
print("Demo: Combined LSA + Entity Grid Intent Detection")
print("=" * 65)

implicit_tests = [
    "In two days I want to travel to Tehran, what clothes should I bring?",
    "I am heading to Berlin this weekend — hot or cold?",
    "I am starving, where can I eat sushi in London?",
    "need to get from the airport to city center in the morning",
    "will I need a coat in Stockholm tomorrow?",
]

for query in implicit_tests:
    dm = LSADialogueManager(get_initial_frames(), SERVICE_HANDLERS)
    print(f'\nUser:  {query}')
    print(f'Agent: {dm.process_input(query)}')
    print('-' * 55)

Demo: Combined LSA + Entity Grid Intent Detection

User:  In two days I want to travel to Tehran, what clothes should I bring?
  [Override] Clothing word detected → intent=weather
  [EntityGrid] slot location = "tehran" (from grid)
  [EntityGrid] slot date = "in two days" (from grid)
  [System] Service activated: Weather
Agent: [Mock] Weather in Tehran (in two days): 10°C | Rainy | Take an umbrella.
-------------------------------------------------------

User:  I am heading to Berlin this weekend — hot or cold?
  [LSA] query='I am heading to Berlin this weekend — hot or cold?'
        scores: weather:0.85  restaurant:0.54  transport:0.01
        → intent: weather (cos=0.85)
  [EntityGrid] slot location = "berlin" (from grid)
  [EntityGrid] slot date = "this weekend" (from grid)
  [System] Service activated: Weather
Agent: [Mock] Weather in Berlin (this weekend): -2°C | Snowy | Wear a heavy coat and gloves.
-------------------------------------------------------

User:  I am starving, 

---
## Section 9: Multi-Step Slot Filling Demo
The output below shows how the system fills slots turn by turn and
calls the appropriate service when all information is available.

In [14]:
print('=' * 60)
print('Demo: Multi-step Slot Filling')
print('=' * 60)

scenarios = [
    {'title': 'Weather in Stockholm tomorrow',
     'msgs':  ['weather', 'stockholm', 'tomorrow']},
    {'title': 'Cheap pizza restaurant in London',
     'msgs':  ['restaurant', 'pizza', 'london', 'cheap']},
    {'title': 'Morning bus from airport',
     'msgs':  ['bus ticket', 'airport', 'city center', 'morning']},
]

for sc in scenarios:
    dm = LSADialogueManager(get_initial_frames(), SERVICE_HANDLERS)
    print(f"\nScenario: {sc['title']}")
    print('-' * 50)
    for msg in sc['msgs']:
        print(f'  User:  {msg}')
        resp = dm.process_input(msg)
        print(f'  Agent: {resp}')


Demo: Multi-step Slot Filling

Scenario: Weather in Stockholm tomorrow
--------------------------------------------------
  User:  weather
  [LSA] query='weather'
        scores: weather:1.00  restaurant:0.01  transport:0.01
        → intent: weather (cos=1.00)
  [System] Service activated: Weather
  Agent: Which city are you asking about?
  User:  stockholm
  [LSA] query='stockholm'
        scores: weather:0.71  restaurant:0.01  transport:0.71
        → intent: transport (cos=0.71)
  [EntityGrid] slot location = "stockholm" (from grid)
  [System] Service activated: Transport
  Agent: Where are you starting from? (e.g. home, airport)
  User:  tomorrow
  [LSA] query='tomorrow'
        scores: weather:1.00  restaurant:0.01  transport:0.01
        → intent: weather (cos=1.00)
  [EntityGrid] slot location = "stockholm" (from grid)
  [EntityGrid] slot date = "tomorrow" (from grid)
  [System] Service activated: Weather
  Agent: [Mock] Weather in Stockholm (tomorrow): 14°C | Cloudy | Wear an 

---
## Section 10: Multilingual Agent — Putting It All Together

The `MultilingualAgent` combines all components into a single pipeline:

```
User Input (any language)
        ↓
  TranslationLayer.to_english()        ← preserves all languages
        ↓
  LSADialogueManager.process_input()
    ├── Step 1: Clothing word override
    ├── Step 2: LSA cosine similarity   ← NEW: distributional semantics
    ├── Step 3: Keyword fallback
    ├── Step 4: EntityGrid carryover    ← NEW: discourse entity tracking
    ├── slot-filling loop
    └── service_handler(slots)         (live OWM API or mock data)
        ↓
  TranslationLayer.from_english()      ← translate back to user's language
        ↓
  Response in user's original language
```

> **Translation is fully preserved:** every input is translated to English before
> processing, and every response is translated back to the user's language.

In [15]:
class MultilingualAgent:
    """Full pipeline: any language → English processing → response in user language."""

    def __init__(self):
        self.translator = TranslationLayer()
        self.dm = SemanticDialogueManager(get_initial_frames(), SERVICE_HANDLERS)

    def chat(self, user_text):
        en_input   = self.translator.to_english(user_text)
        en_response = self.dm.process_input(en_input)
        return self.translator.from_english(en_response)

    def reset(self):
        self.dm = SemanticDialogueManager(get_initial_frames(), SERVICE_HANDLERS)
        self.translator = TranslationLayer()


print('=' * 60)
print('Demo: MultilingualAgent — same question in 4 languages')
print('=' * 60)

same_question = [
    ('Persian',   'فردا می‌خواهم به استکهلم بروم، چه لباسی بپوشم؟'),
    ('Swedish',   'Vad ska jag ha pa mig i Goteborg imorgon?'),
    ('German',    'Was soll ich morgen in Berlin anziehen?'),
    ('Ukrainian', 'Яку погоду очікувати в Стокгольмі завтра?'),
    ('English',   'What should I wear in Stockholm tomorrow?'),
]

for lang, question in same_question:
    agent = MultilingualAgent()
    print(f'\n[{lang}] User:  {question}')
    print(f'[{lang}] Agent: {agent.chat(question)}')
    print('-' * 55)


Demo: MultilingualAgent — same question in 4 languages

[Persian] User:  فردا می‌خواهم به استکهلم بروم، چه لباسی بپوشم؟
  [Translate] Persian → EN: "I want to go to Stockholm tomorrow, what should I wear?"
  [Override] Clothing word detected → intent=weather
  [EntityGrid] slot location = "stockholm" (from grid)
  [EntityGrid] slot date = "tomorrow" (from grid)
  [System] Service activated: Weather
[Persian] Agent: [مسخ] آب و هوا در استکهلم (فردا): 14 درجه سانتی گراد | ابری | یک لایه اضافی بپوشید.
-------------------------------------------------------

[Swedish] User:  Vad ska jag ha pa mig i Goteborg imorgon?
  [Translate] Swedish → EN: "What should I wear in Gothenburg tomorrow?"
  [Override] Clothing word detected → intent=weather
  [EntityGrid] slot date = "tomorrow" (from grid)
  [System] Service activated: Weather
[Swedish] Agent: Vilken stad frågar du om?
-------------------------------------------------------

[German] User:  Was soll ich morgen in Berlin anziehen?
  [Translat

---
## Section 11: Interactive Chat

You can now chat with the full multilingual agent in real time!

- Write in **any supported language** (Persian, Swedish, German, French, Arabic, Spanish, Ukrainian, English)
- Receive the response in **the same language**
- Type `exit` (or `خروج`, `avsluta`) to end the session

> **How to start:** Run the cell below.

### Example session
```
You:   در دو روز دیگر می‌خواهم به تهران بروم، چه لباسی با خودم ببرم؟
  [Translate] Persian → EN: "In two days I want to travel to Tehran, what clothes should I bring?"
  [Override] Clothing word detected → intent=weather
  [EntityGrid] location = "tehran", date = "in two days"
  [System] Service activated: Weather
Agent: آب‌وهوای تهران برای دو روز دیگر: 12 درجه | نیمه‌ابری | لباس متوسط کافی است.

You:   I am starving, find me cheap sushi in London
  [LSA] scores: weather:0.02  restaurant:0.84  transport:0.05
       → intent: restaurant (cos=0.84)
  [System] Service activated: Restaurant
Agent: Best sushi restaurant in London (cheap): Tokyo Garden | Rating: 5/5 ⭐
```

In [ ]:
import sys as _sys, time as _time

def start_interactive_chat():
    """Multilingual chat — optimized for VS Code Jupyter."""
    agent = MultilingualAgent()
    print('Agent: Hello! I understand Persian, Ukrainian, Swedish, Arabic, German, French, Spanish and English.')
    print('Type in any language — type "exit" to quit.')
    print('-' * 50)
    _sys.stdout.flush()

    while True:
        try:
            # We just use plain input to avoid the "You: You:" duplication
            # VS Code handles native input() best when not paired with sleeps
            user_input = input('You: ').strip()
        except EOFError:
            break

        if not user_input:
            continue
            
        # Manually print the user's input because VS Code sometimes hides
        # what was typed in the input box from the final cell output log
        print(f'\n--- You said: {user_input} ---')
            
        if user_input.lower() in ('exit', 'quit', 'bye', 'خروج', 'avsluta', 'вихід'):
            print('Agent: Goodbye!')
            break

        response = agent.chat(user_input)
        print(f'Agent: {response}')
        print() 
        _sys.stdout.flush()

# Run this cell manually when you want to start chatting:
start_interactive_chat()


Agent: Hello! I understand Persian, Ukrainian, Swedish, Arabic, German, French, Spanish and English.
Type in any language — type "exit" to quit.
--------------------------------------------------

--- You said: سلام ---
  [Translate] Persian → EN: "hello"
Agent: سلام! من می توانم در مورد آب و هوا، رستوران ها یا حمل و نقل کمک کنم.


--- You said: من فردا میخواهم به تهران مسافرت کنم . چه لباسی بپوشم؟ ---
  [Translate] Persian → EN: "I want to travel to Tehran tomorrow. What should I wear?"
  [Override] Clothing word detected → intent=weather
  [EntityGrid] slot location = "tehran" (from grid)
  [EntityGrid] slot date = "tomorrow" (from grid)
  [System] Service activated: Weather
Agent: [ساخت] هوای تهران (فردا): 10 درجه سانتی گراد | بارانی | یک چتر بردارید


--- You said: گرسنه هستم ---
  [Translate] Persian → EN: "I am hungry"
  [LSA] query='I am hungry'
        scores: weather:0.01  restaurant:1.00  transport:0.00
        → intent: restaurant (cos=1.00)
  [EntityGrid] slot location = "t